# SPARK Tutorial 2026
## Brain Tumor Segmentation with U-Net (TensorFlow / Keras)

<center><img src="https://www.mayoclinic.org/-/media/kcms/gbs/patient-consumer/images/2014/10/30/15/17/mcdc7_brain_cancer-8col.jpg" alt="Brain MRI" style='width:350px;'></center>

**Image source**: Mayo Clinic

**References**: [Lee et al., 2024](https://doi.org/10.1016/j.dib.2024.111159) | [Ronneberger et al., U-Net, MICCAI 2015](https://arxiv.org/abs/1505.04597)

## About This Tutorial

This hands-on notebook guides you through building a brain tumor segmentation model using **U-Net** in TensorFlow/Keras on the LGG MRI dataset.

> **Environment**: Designed to run on **Kaggle** with GPU acceleration.
> **Dataset**: [LGG MRI Segmentation](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation)

# <p style="background-color:#2c3e50;color:white;font-size:22px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Table of Contents</p>

1. [Introduction & Background](#part1)
2. [Segmentation Metrics](#part2)
3. [U-Net Architecture](#part3)
4. [SOTA Model Overview](#part4)
5. [Hands-On Implementation](#part5)
    - [5.1 Imports](#sec5_1)
    - [5.2 Data Loading](#sec5_2)
    - [5.3 Preprocessing & Augmentation](#sec5_3)
    - [5.4 Build U-Net](#sec5_4)
    - [5.5 Loss & Metrics](#sec5_5)
    - [5.6 Train](#sec5_6)
    - [5.7 Visualization](#sec5_7)
    - [5.8 Evaluation](#sec5_8)
    - [5.9 Prediction](#sec5_9)
6. [Summary](#part6)
7. [References](#refs)

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 1 — Introduction & Background</p> <a id="part1"></a>

A **brain tumor** is an abnormal growth of cells in the brain. **Gliomas** are the most common primary brain tumors. Automated segmentation from MRI helps with surgical planning, radiation therapy, and treatment monitoring.

**Image segmentation** classifies every pixel in an image into a category. For brain tumors, we assign each pixel as **tumor** or **background**.

### Quiz 1: Background Concepts

In [ ]:
# # Quiz 1: Brain Tumor Segmentation Concepts
#
# # Question 1:
# # What is the key difference between image classification and image segmentation?
# # A) Classification assigns a label to each pixel; segmentation assigns one label to the whole image
# # B) Classification assigns one label to the whole image; segmentation assigns a label to each pixel
# # C) They are the same thing
# # D) Classification works on 3D images; segmentation works on 2D images
#
# # Your answer:
# q1_answer = "___"  # Replace with A, B, C, or D
#
# # Question 2:
# # Which MRI sequence suppresses CSF signal and is commonly used for detecting perilesional edema?
# # A) T1-weighted
# # B) T1 with Contrast Enhancement
# # C) T2-weighted
# # D) FLAIR
#
# # Your answer:
# q2_answer = "___"  # Replace with A, B, C, or D
#
# # Question 3:
# # Why is automated brain tumor segmentation important in clinical practice?
# # A) It replaces the need for MRI scans
# # B) It speeds up diagnosis and reduces inter-observer variability
# # C) It eliminates the need for surgery
# # D) It makes tumors disappear
#
# # Your answer:
# q3_answer = "___"  # Replace with A, B, C, or D

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 2 — Segmentation Metrics</p> <a id="part2"></a>

| Metric | Formula | Range | Notes |
|--------|---------|-------|-------|
| **Dice Coefficient** | $\frac{2 \|P \cap G\|}{\|P\| + \|G\|}$ | 0–1 | Primary metric for medical segmentation |
| **IoU (Jaccard)** | $\frac{\|P \cap G\|}{\|P \cup G\|}$ | 0–1 | More strict than Dice |
| **Pixel Accuracy** | $\frac{\text{Correct pixels}}{\text{Total pixels}}$ | 0–1 | Misleading with class imbalance |

**Dice Loss** = $-\text{Dice}(P, G)$ — allows direct optimization of the overlap metric during training.

### Quiz 2: Segmentation Metrics

In [ ]:
# # Quiz 2: Understanding Segmentation Metrics
#
# # Question 1:
# # If a brain image has 2% tumor pixels and 98% background pixels,
# # and a model predicts ALL pixels as background, what is its pixel accuracy?
# # A) 2%
# # B) 50%
# # C) 98%
# # D) 0%
#
# # Your answer:
# q1_answer = "___"
#
# # Question 2:
# # For that same model (predicts all background), what is its Dice coefficient for the tumor class?
# # A) 0.98
# # B) 0.50
# # C) 0.0
# # D) 1.0
#
# # Your answer:
# q2_answer = "___"
#
# # Question 3:
# # Why is Dice coefficient preferred over pixel accuracy for tumor segmentation?
# # A) Dice is easier to compute
# # B) Dice focuses on the positive class and handles class imbalance
# # C) Pixel accuracy is always wrong
# # D) Dice always gives higher values
#
# # Your answer:
# q3_answer = "___"  

In [ ]:
# # Quiz 2 — Coding Challenge: Compute Dice and IoU
#
# # Given two binary masks (flattened), compute the Dice coefficient and IoU manually.
# # Do NOT use any library functions — use basic Python/NumPy operations.
#
# import numpy as np
#
# y_true = np.array([0, 1, 1, 1, 0, 0, 1, 1, 0, 0])
# y_pred = np.array([0, 1, 0, 1, 0, 0, 1, 0, 0, 1])
#
# # TODO: Calculate intersection (number of pixels where both are 1)
# intersection = ___
#
# # TODO: Calculate Dice coefficient: 2 * intersection / (sum(y_true) + sum(y_pred))
# dice = ___
#
# # TODO: Calculate IoU: intersection / (sum(y_true) + sum(y_pred) - intersection)
# iou = ___
#
# print(f"Intersection: {intersection}")
# print(f"Dice Coefficient: {dice:.4f}")
# print(f"IoU: {iou:.4f}")
#
# # Expected output:
# # Intersection: 3
# # Dice Coefficient: 0.6667
# # IoU: 0.5000

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 3 — U-Net Architecture</p> <a id="part3"></a>

<center><img src="https://miro.medium.com/max/1200/1*f7YOaE4TWubwaFF7Z1fzNw.png" alt="U-Net Architecture" style='width:800px;'></center>

U-Net has three main parts:
- **Encoder**: Captures *what* is in the image (downsampling path)
- **Bottleneck**: Deepest, most compressed representation
- **Decoder**: Recovers *where* things are (upsampling path)
- **Skip connections**: Concatenate encoder features to decoder for precise localization

### Quiz 3: U-Net Architecture

In [ ]:
# # Quiz 3: Understanding U-Net
#
# # Question 1:
# # What is the purpose of skip connections in U-Net?
# # A) To speed up training
# # B) To skip over broken layers
# # C) To pass fine spatial details from encoder to decoder
# # D) To reduce the number of parameters
#
# # Your answer:
# q1_answer = "___"
#
# # Question 2:
# # In the encoder path, what operation reduces spatial dimensions?
# # A) Batch Normalization
# # B) Conv2DTranspose
# # C) Max Pooling
# # D) Dropout
#
# # Your answer:
# q2_answer = "___"
#
# # Question 3:
# # What operation is used in the decoder to increase spatial dimensions?
# # A) Max Pooling
# # B) Conv2DTranspose (Transposed Convolution)
# # C) Flatten
# # D) Global Average Pooling
#
# # Your answer:
# q3_answer = "___"
#
# # Question 4:
# # Why does U-Net use a 1x1 convolution as the final layer?
# # A) To increase the number of channels
# # B) To apply max pooling
# # C) To map feature channels to the desired number of output classes
# # D) To add skip connections
#
# # Your answer:
# q4_answer = "___"  

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 4 — State-of-the-Art Models</p> <a id="part4"></a>

| Model | Year | Key Innovation | Architecture |
|-------|------|----------------|-------------|
| **nnU-Net** | 2021 | Self-configuring framework | Auto-adapted U-Net variants |
| **Swin UNETR** | 2021 | Shifted-window Transformer encoder | Swin Transformer + U-Net decoder |
| **MedSAM** | 2024 | Foundation model for medical images | ViT encoder + prompt-based decoder |
| **TransBTS** | 2021 | First Transformer for brain tumor seg | 3D CNN + Transformer bottleneck |
| **Attention U-Net** | 2018 | Attention gates on skip connections | U-Net + learned attention maps |

**nnU-Net** ([GitHub](https://github.com/MIC-DKFZ/nnUNet)) — Isensee et al., *Nature Methods* 2021.
**Swin UNETR** ([GitHub](https://github.com/Project-MONAI/tutorials)) — Hatamizadeh et al., *BrainLes/MICCAI 2021*; Tang et al., *CVPR 2022*.
**MedSAM** ([GitHub](https://github.com/bowang-lab/MedSAM)) — Ma et al., *Nature Communications* 2024. ([Brain-specific](https://github.com/vpulab/med-sam-brain))
**TransBTS** ([GitHub](https://github.com/Wenxuan-1119/TransBTS)) — Wang et al., *MICCAI 2021*.
**Attention U-Net** — Oktay et al., *MIDL 2018*. [arXiv:1804.03999](https://arxiv.org/abs/1804.03999)

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 5 — Hands-On Implementation</p> <a id="part5"></a>

## <span style="color:#1a5276;font-size:22px">5.1 Setup & Imports</span> <a id="sec5_1"></a>

In [ ]:
# import system libs
import os
import time
import random
import pathlib
import itertools
from glob import glob
from tqdm import tqdm_notebook, tnrange

# import data handling tools
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_style('darkgrid')
import matplotlib.pyplot as plt
%matplotlib inline
from skimage.color import rgb2gray
from skimage.morphology import label
from skimage.transform import resize
from sklearn.model_selection import train_test_split
from skimage.io import imread, imshow, concatenate_images

# import Deep learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model, load_model, save_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Input, Activation, BatchNormalization, Dropout, Lambda, Conv2D, Conv2DTranspose, MaxPooling2D, concatenate

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

print('modules loaded')

## <span style="color:#1a5276;font-size:22px">5.2 Data Loading</span> <a id="sec5_2"></a>

### Quiz 4: Data Handling

In [ ]:
# # Quiz 4: Understanding Data Preparation
#
# # The function below creates a DataFrame mapping images to masks.
# # Study the code and answer the questions.
#
# # def create_df(data_dir):
# #     images_paths = []
# #     masks_paths = glob(f'{data_dir}/*/*_mask*')
# #     for i in masks_paths:
# #         images_paths.append(i.replace('_mask', ''))
# #     df = pd.DataFrame(data={'images_paths': images_paths, 'masks_paths': masks_paths})
# #     return df
#
# # Question 1:
# # How does the function find the corresponding image for each mask?
# # A) It searches for files with the same name
# # B) It removes '_mask' from the mask filename to get the image filename
# # C) It uses a lookup table
# # D) It reads a CSV file
#
# # Your answer:
# q1_answer = "___"
#
# # Question 2:
# # What does train_test_split with train_size=0.8 do?
# # A) Uses 80% of data for testing
# # B) Uses 80% of data for training and 20% for the remaining split
# # C) Uses 80% of data for validation
# # D) Randomly selects 80 images
#
# # Your answer:
# q2_answer = "___"  

In [ ]:
# # Quiz 4 — Coding Challenge: Write a Data Split Function
#
# # Complete the function that splits a DataFrame into train (70%), valid (15%), test (15%).
# # Use sklearn's train_test_split.
#
# from sklearn.model_selection import train_test_split
# import pandas as pd
#
# def split_data(df, train_ratio=0.7, valid_ratio=0.15, test_ratio=0.15):
#     '''
#     Split DataFrame into train, valid, test sets.
#
#     Args:
#         df: Input DataFrame
#         train_ratio: Proportion for training (default 0.7)
#         valid_ratio: Proportion for validation (default 0.15)
#         test_ratio: Proportion for testing (default 0.15)
#
#     Returns:
#         train_df, valid_df, test_df
#     '''
#     # Step 1: Split into train and remaining (valid + test)
#     train_df, remaining_df = train_test_split(df, train_size=___)
#
#     # Step 2: Split remaining into valid and test
#     # Hint: valid is valid_ratio / (valid_ratio + test_ratio) of the remaining
#     valid_df, test_df = train_test_split(remaining_df, train_size=___)
#
#     return train_df, valid_df, test_df
#
# # Test your function
# sample_df = pd.DataFrame({'data': range(100)})
# train, valid, test = split_data(sample_df)
# print(f"Train: {len(train)}, Valid: {len(valid)}, Test: {len(test)}")
#
# # Expected output: Train: 70, Valid: 15, Test: 15

In [ ]:
# Function to create a DataFrame mapping images to their masks
def create_df(data_dir):
    images_paths = []
    masks_paths = glob(f'{data_dir}/*/*_mask*')

    for i in masks_paths:
        images_paths.append(i.replace('_mask', ''))

    df = pd.DataFrame(data= {'images_paths': images_paths, 'masks_paths': masks_paths})

    return df

# Function to split DataFrame into train (80%), valid (10%), test (10%)
def split_df(df):
    # create train_df
    train_df, dummy_df = train_test_split(df, train_size= 0.8)

    # create valid_df and test_df
    valid_df, test_df = train_test_split(dummy_df, train_size= 0.5)

    return train_df, valid_df, test_df

## <span style="color:#1a5276;font-size:22px">5.3 Preprocessing & Augmentation</span> <a id="sec5_3"></a>

### Quiz 5: Data Augmentation

In [ ]:
# # Quiz 5: Understanding Data Augmentation
#
# # Question 1:
# # Why do we apply the SAME augmentation to both the image and its mask?
# # A) To save computation time
# # B) To ensure the spatial correspondence between image and mask is preserved
# # C) Because masks don't need augmentation
# # D) To make the images look different from the masks
#
# # Your answer:
# q1_answer = "___"
#
# # Question 2:
# # Why do we normalize images to [0, 1] range?
# # A) It makes images look better
# # B) It helps neural networks converge faster and more stably
# # C) It is required by the GPU
# # D) It reduces file size
#
# # Your answer:
# q2_answer = "___"
#
# # Question 3:
# # Why do we NOT apply augmentation to validation and test sets?
# # A) To save time
# # B) Because augmentation corrupts the data
# # C) To evaluate the model on realistic, unmodified data
# # D) Augmentation only works on training data
#
# # Your answer:
# q3_answer = "___"  

In [ ]:
# # Quiz 5 — Coding Challenge: Define an Augmentation Pipeline
#
# # Create an augmentation dictionary for ImageDataGenerator that includes:
# #   - rotation up to 15 degrees
# #   - width and height shifts of 10%
# #   - zoom range of 20%
# #   - horizontal AND vertical flips
# #   - fill_mode 'reflect'
# #
# # Then create a SECOND dictionary for the validation set.
#
# # TODO: Define training augmentation dictionary
# train_aug = dict(
#     ___
# )
#
# # TODO: Define validation augmentation dictionary
# # Hint: What augmentation should we apply to validation data?
# valid_aug = ___
#
# print("Training augmentation keys:", list(train_aug.keys()))
# print("Validation augmentation:", valid_aug)

In [ ]:
def create_gens(df, aug_dict):
    img_size = (256, 256)
    batch_size = 40


    img_gen = ImageDataGenerator(**aug_dict)
    msk_gen = ImageDataGenerator(**aug_dict)

    # Create general generator
    image_gen = img_gen.flow_from_dataframe(df, x_col='images_paths', class_mode=None, color_mode='rgb', target_size=img_size,
                                            batch_size=batch_size, save_to_dir=None, save_prefix='image', seed=1)

    mask_gen = msk_gen.flow_from_dataframe(df, x_col='masks_paths', class_mode=None, color_mode='grayscale', target_size=img_size,
                                            batch_size=batch_size, save_to_dir=None, save_prefix= 'mask', seed=1)

    gen = zip(image_gen, mask_gen)

    for (img, msk) in gen:
        img = img / 255
        msk = msk / 255
        msk[msk > 0.5] = 1
        msk[msk <= 0.5] = 0

        yield (img, msk)

In [ ]:
def show_images(images, masks):
    plt.figure(figsize=(12, 12))
    for i in range(25):
        plt.subplot(5, 5, i+1)
        img_path = images[i]
        mask_path = masks[i]
        # read image and convert it to RGB scale
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # read mask
        mask = cv2.imread(mask_path)
        # show image and mask
        plt.imshow(image)
        plt.imshow(mask, alpha=0.4)

        plt.axis('off')

    plt.tight_layout()
    plt.show()

### Load Data and Create Generators

In [ ]:
# Auto-detect dataset path (varies depending on how the dataset was added on Kaggle)
import os
candidate_paths = [
    '/kaggle/input/lgg-mri-segmentation/kaggle_3m',
    '/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m',
    '/kaggle/input/lgg-mri-segmentation',
    '/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation',
]
data_dir = None
for p in candidate_paths:
    if os.path.isdir(p):
        data_dir = p
        break
if data_dir is None:
    # Fallback: search for any directory containing '_mask' files
    for root, dirs, files in os.walk('/kaggle/input'):
        if any('_mask' in f for f in files):
            data_dir = os.path.dirname(root)
            break
assert data_dir is not None, "Dataset not found! Please check the dataset path."
print(f"Using dataset path: {data_dir}")

df = create_df(data_dir)
train_df, valid_df, test_df = split_df(df)

print(f"Total images: {len(df)}")
print(f"Training:     {len(train_df)}")
print(f"Validation:   {len(valid_df)}")
print(f"Test:         {len(test_df)}")

tr_aug_dict = dict(rotation_range=0.2,
                            width_shift_range=0.05,
                            height_shift_range=0.05,
                            shear_range=0.05,
                            zoom_range=0.05,
                            horizontal_flip=True,
                            fill_mode='nearest')


train_gen = create_gens(train_df, aug_dict=tr_aug_dict)
valid_gen = create_gens(valid_df, aug_dict={})
test_gen = create_gens(test_df, aug_dict={})

show_images(list(train_df['images_paths']), list(train_df['masks_paths']))

## <span style="color:#1a5276;font-size:22px">5.4 Build U-Net Model</span> <a id="sec5_4"></a>

### Quiz 6: U-Net Implementation

In [ ]:
# # Quiz 6: Understanding the U-Net Code
#
# # Study the U-Net function below and answer the questions.
#
# # Question 1:
# # The encoder uses Conv2D → ReLU → Conv2D → BatchNorm → ReLU → MaxPool.
# # What does BatchNormalization do?
# # A) Adds random noise to features
# # B) Normalizes activations to have zero mean and unit variance, stabilizing training
# # C) Removes features with low values
# # D) Reduces the number of channels
#
# # Your answer:
# q1_answer = "___"
#
# # Question 2:
# # In the decoder, what does the 'concatenate' operation do?
# # A) Adds the feature maps element-wise
# # B) Multiplies the feature maps
# # C) Stacks the feature maps along the channel axis
# # D) Averages the feature maps
#
# # Your answer:
# q2_answer = "___"
#
# # Question 3:
# # Why does the final Conv2D layer use kernel_size=(1,1) and activation="sigmoid"?
# # A) 1x1 convolution maps to 1 output channel (binary mask); sigmoid outputs probabilities in [0,1]
# # B) 1x1 is faster than 3x3
# # C) Sigmoid makes the output negative
# # D) It adds skip connections
#
# # Your answer:
# q3_answer = "___"  

In [ ]:
# # Quiz 6 — Coding Challenge: Build a U-Net Encoder Block
#
# # Complete the function that creates ONE encoder block for U-Net.
# # Each block should have: Conv2D -> ReLU -> Conv2D -> BatchNorm -> ReLU -> MaxPool
# # Return both the output (after pooling) AND the skip connection (before pooling).
#
# from tensorflow.keras.layers import (Input, Conv2D, Activation,
#     BatchNormalization, MaxPooling2D)
#
# def encoder_block(inputs, num_filters):
#     '''
#     Creates one encoder block.
#
#     Args:
#         inputs: Input tensor
#         num_filters: Number of convolution filters
#
#     Returns:
#         skip: Feature map BEFORE pooling (for skip connection)
#         pool: Feature map AFTER pooling (input to next block)
#     '''
#     # TODO: First convolution (3x3, same padding)
#     x = Conv2D(filters=___, kernel_size=___, padding=___)(inputs)
#     x = Activation("relu")(x)
#
#     # TODO: Second convolution (3x3, same padding)
#     x = Conv2D(filters=___, kernel_size=___, padding=___)(x)
#     x = BatchNormalization(axis=3)(x)
#     x = Activation("relu")(x)
#
#     # TODO: Save skip connection BEFORE pooling
#     skip = ___
#
#     # TODO: Apply max pooling (2x2)
#     pool = MaxPooling2D(pool_size=___)(x)
#
#     return skip, pool
#
# # Test your block
# test_input = Input(shape=(256, 256, 3))
# skip, pool = encoder_block(test_input, 64)
# print(f"Skip connection shape: {skip.shape}")
# print(f"After pooling shape:   {pool.shape}")
#
# # Expected output:
# # Skip connection shape: (None, 256, 256, 64)
# # After pooling shape:   (None, 128, 128, 64)

In [ ]:
def unet(input_size=(256, 256, 3)):
    inputs = Input(input_size)

    # First DownConvolution / Encoder Leg will begin, so start with Conv2D
    conv1 = Conv2D(filters=64, kernel_size=(3, 3), padding="same")(inputs)
    bn1 = Activation("relu")(conv1)
    conv1 = Conv2D(filters=64, kernel_size=(3, 3), padding="same")(bn1)
    bn1 = BatchNormalization(axis=3)(conv1)
    bn1 = Activation("relu")(bn1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(bn1)

    conv2 = Conv2D(filters=128, kernel_size=(3, 3), padding="same")(pool1)
    bn2 = Activation("relu")(conv2)
    conv2 = Conv2D(filters=128, kernel_size=(3, 3), padding="same")(bn2)
    bn2 = BatchNormalization(axis=3)(conv2)
    bn2 = Activation("relu")(bn2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(bn2)

    conv3 = Conv2D(filters=256, kernel_size=(3, 3), padding="same")(pool2)
    bn3 = Activation("relu")(conv3)
    conv3 = Conv2D(filters=256, kernel_size=(3, 3), padding="same")(bn3)
    bn3 = BatchNormalization(axis=3)(conv3)
    bn3 = Activation("relu")(bn3)
    pool3 = MaxPooling2D(pool_size=(2, 2))(bn3)

    conv4 = Conv2D(filters=512, kernel_size=(3, 3), padding="same")(pool3)
    bn4 = Activation("relu")(conv4)
    conv4 = Conv2D(filters=512, kernel_size=(3, 3), padding="same")(bn4)
    bn4 = BatchNormalization(axis=3)(conv4)
    bn4 = Activation("relu")(bn4)
    pool4 = MaxPooling2D(pool_size=(2, 2))(bn4)

    conv5 = Conv2D(filters=1024, kernel_size=(3, 3), padding="same")(pool4)
    bn5 = Activation("relu")(conv5)
    conv5 = Conv2D(filters=1024, kernel_size=(3, 3), padding="same")(bn5)
    bn5 = BatchNormalization(axis=3)(conv5)
    bn5 = Activation("relu")(bn5)

    # Now UpConvolution / Decoder Leg will begin, so start with Conv2DTranspose
    # The gray arrows (in the above image) indicate the skip connections that concatenate
    # the encoder feature map with the decoder, which helps the backward flow of gradients
    # for improved training.
    # After every concatenation we again apply two consecutive regular convolutions so that
    # the model can learn to assemble a more precise output.

    up6 = concatenate([Conv2DTranspose(512, kernel_size=(2, 2), strides=(2, 2), padding="same")(bn5), conv4], axis=3)
    conv6 = Conv2D(filters=512, kernel_size=(3, 3), padding="same")(up6)
    bn6 = Activation("relu")(conv6)
    conv6 = Conv2D(filters=512, kernel_size=(3, 3), padding="same")(bn6)
    bn6 = BatchNormalization(axis=3)(conv6)
    bn6 = Activation("relu")(bn6)

    up7 = concatenate([Conv2DTranspose(256, kernel_size=(2, 2), strides=(2, 2), padding="same")(bn6), conv3], axis=3)
    conv7 = Conv2D(filters=256, kernel_size=(3, 3), padding="same")(up7)
    bn7 = Activation("relu")(conv7)
    conv7 = Conv2D(filters=256, kernel_size=(3, 3), padding="same")(bn7)
    bn7 = BatchNormalization(axis=3)(conv7)
    bn7 = Activation("relu")(bn7)

    up8 = concatenate([Conv2DTranspose(128, kernel_size=(2, 2), strides=(2, 2), padding="same")(bn7), conv2], axis=3)
    conv8 = Conv2D(filters=128, kernel_size=(3, 3), padding="same")(up8)
    bn8 = Activation("relu")(conv8)
    conv8 = Conv2D(filters=128, kernel_size=(3, 3), padding="same")(bn8)
    bn8 = BatchNormalization(axis=3)(conv8)
    bn8 = Activation("relu")(bn8)

    up9 = concatenate([Conv2DTranspose(64, kernel_size=(2, 2), strides=(2, 2), padding="same")(bn8), conv1], axis=3)
    conv9 = Conv2D(filters=64, kernel_size=(3, 3), padding="same")(up9)
    bn9 = Activation("relu")(conv9)
    conv9 = Conv2D(filters=64, kernel_size=(3, 3), padding="same")(bn9)
    bn9 = BatchNormalization(axis=3)(conv9)
    bn9 = Activation("relu")(bn9)

    conv10 = Conv2D(filters=1, kernel_size=(1, 1), activation="sigmoid")(bn9)

    return Model(inputs=[inputs], outputs=[conv10])

## <span style="color:#1a5276;font-size:22px">5.5 Loss Functions & Metrics</span> <a id="sec5_5"></a>

In [ ]:
# Dice coefficient: measures overlap between prediction and ground truth
def dice_coef(y_true, y_pred, smooth=100):
    y_true_flatten = K.flatten(y_true)
    y_pred_flatten = K.flatten(y_pred)

    intersection = K.sum(y_true_flatten * y_pred_flatten)
    union = K.sum(y_true_flatten) + K.sum(y_pred_flatten)
    return (2 * intersection + smooth) / (union + smooth)

# Dice loss: negative Dice coefficient (to minimize)
def dice_loss(y_true, y_pred, smooth=100):
    return -dice_coef(y_true, y_pred, smooth)

# IoU coefficient: Intersection over Union
def iou_coef(y_true, y_pred, smooth=100):
    intersection = K.sum(y_true * y_pred)
    sum = K.sum(y_true + y_pred)
    iou = (intersection + smooth) / (sum - intersection + smooth)
    return iou

## <span style="color:#1a5276;font-size:22px">5.6 Compile & Train</span> <a id="sec5_6"></a>

### Quiz 7: Training Concepts

In [ ]:
# # Quiz 7: Understanding Model Training
#
# # Question 1:
# # What does ModelCheckpoint with save_best_only=True do?
# # A) Saves the model after every epoch
# # B) Saves only the model with the best validation performance
# # C) Saves only the final model
# # D) Prevents the model from being saved
#
# # Your answer:
# q1_answer = "___"
#
# # Question 2:
# # Why do we use steps_per_epoch = len(train_df) / batch_size?
# # A) To make training faster
# # B) It defines how many batches constitute one complete pass through the training data
# # C) It controls the learning rate
# # D) It sets the number of epochs
#
# # Your answer:
# q2_answer = "___"
#
# # Question 3:
# # If training Dice keeps improving but validation Dice starts decreasing, what is happening?
# # A) The model is underfitting
# # B) The model is overfitting — memorizing training data instead of generalizing
# # C) The learning rate is too low
# # D) The batch size is too large
#
# # Your answer:
# q3_answer = "___"  

In [ ]:
# # Quiz 7 — Coding Challenge: Implement Dice Loss from Scratch
#
# # Implement the Dice coefficient and Dice loss using TensorFlow/Keras backend.
# # Then use them to compile a simple model.
#
# import tensorflow as tf
# from tensorflow.keras import backend as K
#
# def my_dice_coef(y_true, y_pred, smooth=1.0):
#     '''
#     Compute Dice coefficient.
#
#     Steps:
#       1. Flatten both tensors
#       2. Compute intersection = sum(y_true * y_pred)
#       3. Dice = (2 * intersection + smooth) / (sum(y_true) + sum(y_pred) + smooth)
#     '''
#     # TODO: Flatten both tensors
#     y_true_f = K.flatten(___)
#     y_pred_f = K.flatten(___)
#
#     # TODO: Compute intersection
#     intersection = K.sum(___)
#
#     # TODO: Compute and return Dice coefficient
#     dice = (2 * ___ + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)
#     return dice
#
# def my_dice_loss(y_true, y_pred):
#     '''
#     Dice loss = negative Dice coefficient (since we minimize loss).
#     '''
#     # TODO: Return negative dice coefficient
#     return ___
#
# # Test: Verify with known values
# y_t = tf.constant([[1.0, 1.0, 0.0, 0.0]])
# y_p = tf.constant([[1.0, 0.8, 0.1, 0.0]])
# print(f"Dice coef: {my_dice_coef(y_t, y_p, smooth=1.0).numpy():.4f}")
# print(f"Dice loss: {my_dice_loss(y_t, y_p).numpy():.4f}")

In [ ]:
model = unet()
model.compile(Adamax(learning_rate= 0.001), loss= dice_loss, metrics= ['accuracy', iou_coef, dice_coef])

model.summary()

In [ ]:
epochs = 120
batch_size = 40
callbacks = [ModelCheckpoint('unet.keras', verbose=0, save_best_only=True)]

history = model.fit(train_gen,
                    steps_per_epoch=len(train_df) // batch_size,
                    epochs=epochs,
                    verbose=1,
                    callbacks=callbacks,
                    validation_data = valid_gen,
                    validation_steps=len(valid_df) // batch_size)

## <span style="color:#1a5276;font-size:22px">5.7 Training History Visualization</span> <a id="sec5_7"></a>

In [ ]:
def plot_training(hist):
    '''
    This function take training model and plot history of accuracy and losses with the best epoch in both of them.
    '''

    # Define needed variables
    tr_acc = hist.history['accuracy']
    tr_iou = hist.history['iou_coef']
    tr_dice = hist.history['dice_coef']
    tr_loss = hist.history['loss']

    val_acc = hist.history['val_accuracy']
    val_iou = hist.history['val_iou_coef']
    val_dice = hist.history['val_dice_coef']
    val_loss = hist.history['val_loss']

    index_acc = np.argmax(val_acc)
    acc_highest = val_acc[index_acc]
    index_iou = np.argmax(val_iou)
    iou_highest = val_iou[index_iou]
    index_dice = np.argmax(val_dice)
    dice_highest = val_dice[index_dice]
    index_loss = np.argmin(val_loss)
    val_lowest = val_loss[index_loss]

    Epochs = [i+1 for i in range(len(tr_acc))]

    acc_label = f'best epoch= {str(index_acc + 1)}'
    iou_label = f'best epoch= {str(index_iou + 1)}'
    dice_label = f'best epoch= {str(index_dice + 1)}'
    loss_label = f'best epoch= {str(index_loss + 1)}'

    # Plot training history
    plt.figure(figsize= (20, 20))
    plt.style.use('fivethirtyeight')

    # Training Accuracy
    plt.subplot(2, 2, 1)
    plt.plot(Epochs, tr_acc, 'r', label= 'Training Accuracy')
    plt.plot(Epochs, val_acc, 'g', label= 'Validation Accuracy')
    plt.scatter(index_acc + 1 , acc_highest, s= 150, c= 'blue', label= acc_label)
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # Training IoU
    plt.subplot(2, 2, 2)
    plt.plot(Epochs, tr_iou, 'r', label= 'Training IoU')
    plt.plot(Epochs, val_iou, 'g', label= 'Validation IoU')
    plt.scatter(index_iou + 1 , iou_highest, s= 150, c= 'blue', label= iou_label)
    plt.title('Training and Validation IoU Coefficient')
    plt.xlabel('Epochs')
    plt.ylabel('IoU')
    plt.legend()

    # Training Dice
    plt.subplot(2, 2, 3)
    plt.plot(Epochs, tr_dice, 'r', label= 'Training Dice')
    plt.plot(Epochs, val_dice, 'g', label= 'Validation Dice')
    plt.scatter(index_dice + 1 , dice_highest, s= 150, c= 'blue', label= dice_label)
    plt.title('Training and Validation Dice Coefficient')
    plt.xlabel('Epochs')
    plt.ylabel('Dice')
    plt.legend()

    # Training Loss
    plt.subplot(2, 2, 4)
    plt.plot(Epochs, tr_loss, 'r', label= 'Training loss')
    plt.plot(Epochs, val_loss, 'g', label= 'Validation loss')
    plt.scatter(index_loss + 1, val_lowest, s= 150, c= 'blue', label= loss_label)
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout
    plt.show()

In [ ]:
plot_training(history)

## <span style="color:#1a5276;font-size:22px">5.8 Model Evaluation</span> <a id="sec5_8"></a>

In [ ]:
ts_length = len(test_df)
test_batch_size = max(sorted([ts_length // n for n in range(1, ts_length + 1) if ts_length%n == 0 and ts_length/n <= 80]))
test_steps = ts_length // test_batch_size

train_score = model.evaluate(train_gen, steps= test_steps, verbose= 1)
valid_score = model.evaluate(valid_gen, steps= test_steps, verbose= 1)
test_score = model.evaluate(test_gen, steps= test_steps, verbose= 1)


print("Train Loss: ", train_score[0])
print("Train Accuracy: ", train_score[1])
print("Train IoU: ", train_score[2])
print("Train Dice: ", train_score[3])
print('-' * 20)

print("Valid Loss: ", valid_score[0])
print("Valid Accuracy: ", valid_score[1])
print("Valid IoU: ", valid_score[2])
print("Valid Dice: ", valid_score[3])
print('-' * 20)

print("Test Loss: ", test_score[0])
print("Test Accuracy: ", test_score[1])
print("Test IoU: ", test_score[2])
print("Test Dice: ", test_score[3])

## <span style="color:#1a5276;font-size:22px">5.9 Prediction Visualization</span> <a id="sec5_9"></a>

In [ ]:
for _ in range(20):
    index = np.random.randint(1, len(test_df.index))
    img = cv2.imread(test_df['images_paths'].iloc[index])
    img = cv2.resize(img, (256, 256))
    img = img/255
    img = img[np.newaxis, :, :, : ]

    predicted_img = model.predict(img)

    plt.figure(figsize=(12, 12))

    plt.subplot(1, 3, 1)
    plt.imshow(np.squeeze(img))
    plt.axis('off')
    plt.title('Original Image')

    plt.subplot(1, 3, 2)
    plt.imshow(np.squeeze(cv2.imread(test_df['masks_paths'].iloc[index])))
    plt.axis('off')
    plt.title('Original Mask')

    plt.subplot(1, 3, 3)
    plt.imshow(np.squeeze(predicted_img) > 0.5 )
    plt.title('Prediction')
    plt.axis('off')

    plt.show()

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 6 — Summary</p> <a id="part6"></a>

In this tutorial we:
1. Learned about brain tumors and why automated segmentation matters
2. Understood key metrics: Dice, IoU, and pixel accuracy
3. Deep-dived into the U-Net architecture
4. Surveyed SOTA models: nnU-Net, Swin UNETR, MedSAM, TransBTS, Attention U-Net
5. Built a complete U-Net pipeline achieving ~89% Dice on the LGG MRI dataset

# <p style="background-color:#c0392b;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Quiz Answer Key</p>

<details>
<summary><b>Click to reveal answers — Multiple Choice</b></summary>

**Quiz 1: Background Concepts**
1. **B** — Classification assigns one label to the whole image; segmentation assigns a label to each pixel
2. **D** — FLAIR
3. **B** — It speeds up diagnosis and reduces inter-observer variability

**Quiz 2: Segmentation Metrics**
1. **C** — 98% (correctly classifies all background pixels)
2. **C** — 0.0 (no overlap with any tumor pixel)
3. **B** — Dice focuses on the positive class and handles class imbalance

**Quiz 3: U-Net Architecture**
1. **C** — To pass fine spatial details from encoder to decoder
2. **C** — Max Pooling
3. **B** — Conv2DTranspose (Transposed Convolution)
4. **C** — To map feature channels to the desired number of output classes

**Quiz 4: Data Handling**
1. **B** — It removes '_mask' from the mask filename to get the image filename
2. **B** — Uses 80% of data for training and 20% for the remaining split

**Quiz 5: Data Augmentation**
1. **B** — To ensure the spatial correspondence between image and mask is preserved
2. **B** — It helps neural networks converge faster and more stably
3. **C** — To evaluate the model on realistic, unmodified data

**Quiz 6: U-Net Implementation**
1. **B** — Normalizes activations to have zero mean and unit variance, stabilizing training
2. **C** — Stacks the feature maps along the channel axis
3. **A** — 1x1 convolution maps to 1 output channel (binary mask); sigmoid outputs probabilities in [0,1]

**Quiz 7: Training Concepts**
1. **B** — Saves only the model with the best validation performance
2. **B** — It defines how many batches constitute one complete pass through the training data
3. **B** — The model is overfitting — memorizing training data instead of generalizing

</details>

<details>
<summary><b>Click to reveal answers — Coding Challenges</b></summary>

**Quiz 2 — Coding: Dice and IoU**
```python
intersection = np.sum(y_true * y_pred)          # 3
dice = 2 * intersection / (np.sum(y_true) + np.sum(y_pred))  # 0.6667
iou = intersection / (np.sum(y_true) + np.sum(y_pred) - intersection)  # 0.5
```

**Quiz 4 — Coding: Data Split**
```python
train_df, remaining_df = train_test_split(df, train_size=train_ratio)
valid_df, test_df = train_test_split(remaining_df, train_size=valid_ratio / (valid_ratio + test_ratio))
```

**Quiz 5 — Coding: Augmentation Pipeline**
```python
train_aug = dict(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='reflect'
)
valid_aug = {}   # No augmentation for validation
```

**Quiz 6 — Coding: Encoder Block**
```python
x = Conv2D(filters=num_filters, kernel_size=(3, 3), padding="same")(inputs)
x = Activation("relu")(x)
x = Conv2D(filters=num_filters, kernel_size=(3, 3), padding="same")(x)
x = BatchNormalization(axis=3)(x)
x = Activation("relu")(x)
skip = x
pool = MaxPooling2D(pool_size=(2, 2))(x)
```

**Quiz 7 — Coding: Dice Loss**
```python
y_true_f = K.flatten(y_true)
y_pred_f = K.flatten(y_pred)
intersection = K.sum(y_true_f * y_pred_f)
dice = (2 * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)
# my_dice_loss returns: -my_dice_coef(y_true, y_pred)
```

</details>

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">References</p> <a id="refs"></a>

1. **U-Net**: Ronneberger et al., MICCAI 2015. [arXiv:1505.04597](https://arxiv.org/abs/1505.04597)
2. **nnU-Net**: Isensee et al., *Nature Methods* 2021. [DOI:10.1038/s41592-020-01008-z](https://doi.org/10.1038/s41592-020-01008-z) | [GitHub](https://github.com/MIC-DKFZ/nnUNet)
3. **Swin UNETR**: Hatamizadeh et al., BrainLes/MICCAI 2021. [arXiv:2201.01266](https://arxiv.org/abs/2201.01266) | [GitHub](https://github.com/Project-MONAI/tutorials)
4. **Swin UNETR Pre-training**: Tang et al., CVPR 2022. [arXiv:2111.14791](https://arxiv.org/abs/2111.14791)
5. **MedSAM**: Ma et al., *Nature Communications* 2024. [DOI:10.1038/s41467-024-44824-z](https://doi.org/10.1038/s41467-024-44824-z) | [GitHub](https://github.com/bowang-lab/MedSAM) | [Brain](https://github.com/vpulab/med-sam-brain)
6. **TransBTS**: Wang et al., MICCAI 2021. [DOI:10.1007/978-3-030-87193-2_11](https://doi.org/10.1007/978-3-030-87193-2_11) | [GitHub](https://github.com/Wenxuan-1119/TransBTS)
7. **Attention U-Net**: Oktay et al., MIDL 2018. [arXiv:1804.03999](https://arxiv.org/abs/1804.03999)
8. **Regional-Aware U-Net**: Abbas et al. [GitHub](https://github.com/abbas695/Regional_aware_U-NET)
9. **LGG Dataset**: Buda et al., 2019. [Kaggle](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation)
10. **BraTS Challenge**: Menze et al., IEEE TMI, 2015. [DOI:10.1109/TMI.2014.2377694](https://doi.org/10.1109/TMI.2014.2377694)